## Install

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, subprocess, sys
from pathlib import Path

def pip(args):
    print('pip', ' '.join(args))
    subprocess.check_call([sys.executable, '-m', 'pip'] + args)

def tv_pkg():
    import torch
    v = '.'.join(torch.__version__.split('+')[0].split('.')[:2])
    return {'2.8': 'torchvision==0.23.0', '2.9': 'torchvision==0.24.0', '2.10': 'torchvision==0.25.0', '2.11': 'torchvision==0.26.0'}.get(v, 'torchvision')

pip(['uninstall', '-y', 'torchao'])

marker = Path('/content/drive/MyDrive/experiments_vbert/.deps_colmodernvbert_v8')

if not marker.exists():
    pkgs = [
        'numpy==1.26.4',
        'pandas==2.2.2',
        'datasets==3.6.0',
        'Pillow==11.3.0',
        'tqdm==4.67.1',
        'pyarrow==19.0.1',
        'safetensors==0.5.3',
        'fsspec[http]==2025.3.0',
        'huggingface_hub>=1.5.0,<2.0',
        'tokenizers>=0.22.0,<=0.23.0',
        'requests==2.32.4',
        'peft',
    ]
    pip(['install', '-q', '--no-cache-dir'] + pkgs)
    pip(['install', '-q', '--no-cache-dir', '--no-deps', 'git+https://github.com/huggingface/transformers'])
    pip(['install', '-q', '--no-cache-dir', '--no-deps', tv_pkg()])
    marker.parent.mkdir(parents=True, exist_ok=True)
    marker.write_text('ok', encoding='utf-8')
    os.kill(os.getpid(), 9)

print('deps ok')

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from datasets import load_dataset, get_dataset_split_names, load_from_disk
from transformers import Idefics3ImageProcessor, ColModernVBertForRetrieval, ColModernVBertProcessor, AutoTokenizer, AutoModel

print('torch', torch.__version__)
print('numpy', np.__version__, 'pandas', pd.__version__)


pip uninstall -y torchao
deps ok
torch 2.10.0+cu128
numpy 1.26.4 pandas 2.2.2


# Experiments

In [3]:
import os, json, time, math, random, zipfile

RUN_MWS = True
RUN_COCO = True
RUN_VIDORE_VAL = False
RUN_EFFICIENCY = True
RUN_EXTRA_EXPERIMENTS = False

# убрать
SMOKE_TEST = False
MAX_QUERIES = None
MAX_CANDIDATES = None

SEED = 42
MODEL_ID = 'ModernVBERT/colmodernvbert-merged'
MWS_ZIP = Path('/content/drive/MyDrive/eval/mws_vision_retrieval_outputs.zip')
VIDORE_DIR = Path('/content/drive/MyDrive/vidore_ru_work/hf_datasets/train')
VIDORE_JSONL = Path('/content/drive/MyDrive/vidore_ru_work/translated_metadata/train.jsonl')

COCO_ID = 'romrawinjp/multilingual-coco'
COCO_SPLIT = 'train'
COCO_TEST_FRACTION = 0.005

IMAGE_BATCH_SIZE = 4
QUERY_BATCH_SIZE = 4
TOPK_SAVE = 20

OCR_DENSE_CACHE = '/content/drive/MyDrive/experiments_vbert/training_runs/OCR-DENSE/20260512_031525/mws_ocr_pages.parquet'  # example: '/content/drive/MyDrive/.../mws_ocr_pages.parquet'
OCR_DENSE_MODEL = 'intfloat/multilingual-e5-large'
TEXT_BATCH_SIZE = 16

random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

run_id = time.strftime('%Y%m%d_%H%M%S') + ('_smoke' if SMOKE_TEST else '')
root = Path('/content/drive/MyDrive/experiments_vbert')
out = root / 'runs' / run_id
local = Path('/content/experiments_vbert') / run_id
data_dir = local / 'datasets'
emb_dir = local / 'embeddings'
out.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)
emb_dir.mkdir(parents=True, exist_ok=True)

cfg = {
    'run_id': run_id,
    'smoke_test': SMOKE_TEST,
    'max_queries': MAX_QUERIES,
    'max_candidates': MAX_CANDIDATES,
    'model_id': MODEL_ID,
    'mws_zip': str(MWS_ZIP),
    'coco_id': COCO_ID,
    'coco_split': COCO_SPLIT,
    'coco_test_fraction': COCO_TEST_FRACTION,
    'image_batch_size': IMAGE_BATCH_SIZE,
    'query_batch_size': QUERY_BATCH_SIZE,
}
(out / 'config.json').write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf-8')
print('drive out:', out)
print('local work:', local)


drive out: /content/drive/MyDrive/experiments_vbert/runs/20260512_113733
local work: /content/experiments_vbert/20260512_113733


## Tiny helpers

In [4]:
models = [
    dict(id='ZS-MM', kind='colmodern', model_id=MODEL_ID, enabled=True),
    dict(id='RU-LoRA-L2-H2', kind='colmodern_lora', base_model=MODEL_ID, adapter_dir='/content/drive/MyDrive/experiments_vbert/training_runs/RU-LoRA-L2-H2/20260512_071516/adapter/', tokenizer_dir=None, enabled=True),
    dict(id='RU-LoRA-R8-L2', kind='colmodern_lora', base_model=MODEL_ID, adapter_dir='/content/drive/MyDrive/experiments_vbert/training_runs/RU-LoRA-R8-L2/20260512_071458/adapter/', tokenizer_dir=None, enabled=True),
    dict(id='OCR-DENSE', kind='ocr_dense', ocr_cache=OCR_DENSE_CACHE, text_model=OCR_DENSE_MODEL, enabled=True),
]
(out / 'model_registry.json').write_text(json.dumps(models, ensure_ascii=False, indent=2), encoding='utf-8')

def key(x):
    return str(x).replace('/', '_')[:100]

def save_img(img, p):
    p.parent.mkdir(parents=True, exist_ok=True)
    if not p.exists():
        img.convert('RGB').save(p)
    return str(p)

def limit(q, c):
    if not SMOKE_TEST:
        return q.reset_index(drop=True), c.reset_index(drop=True)
    ids = set(c['candidate_id'].head(MAX_CANDIDATES))
    c = c[c['candidate_id'].isin(ids)].reset_index(drop=True)
    q = q[q['positive_id'].isin(ids)].head(MAX_QUERIES).reset_index(drop=True)
    return q, c


## Load data

In [5]:
def load_mws():
    if not MWS_ZIP.exists():
        raise FileNotFoundError(MWS_ZIP)
    d = data_dir / 'mws'
    d.mkdir(parents=True, exist_ok=True)
    if not (d / 'pairs.jsonl').exists():
        with zipfile.ZipFile(MWS_ZIP) as z:
            z.extractall(d)
    pairs = pd.read_json(d / 'pairs.jsonl', lines=True)
    pairs = pairs.drop_duplicates('query_id').sort_values('query_id').reset_index(drop=True)
    pairs['image_path'] = pairs['image_id'].map(lambda x: str(d / 'images' / f'{x}.png'))

    c = pairs[['image_id', 'image_path']].drop_duplicates('image_id')
    c = c.sort_values('image_id').rename(columns={'image_id': 'candidate_id'})
    c.insert(0, 'benchmark', 'mws')

    q = pairs.rename(columns={'image_id': 'positive_id'})
    q.insert(0, 'benchmark', 'mws')
    q = q[['benchmark', 'query_id', 'query', 'positive_id']]

    q, c = limit(q, c)
    print('mws', len(q), 'queries', len(c), 'candidates')
    return dict(name='mws', q=q, c=c)

def load_coco():
    ds = load_dataset(COCO_ID, split=COCO_SPLIT).shuffle(seed=SEED)
    n = max(1, int(len(ds) * COCO_TEST_FRACTION))
    if SMOKE_TEST:
        n = min(MAX_CANDIDATES, len(ds))
    ds = ds.select(range(n))

    rows_c, rows_q = [], []
    img_dir = data_dir / 'coco_ru' / COCO_SPLIT
    empty_ru = 0
    total_ru = 0

    for i, r in enumerate(tqdm(ds, desc='coco')):
        ru = r.get('ru') or []
        if isinstance(ru, str):
            ru = [ru]
        ru = [str(x).strip() for x in ru if str(x).strip()]
        total_ru += len(ru)
        if not ru:
            empty_ru += 1
            continue

        cid = r['cocoid']
        fn = r.get('filename', f'{cid}.jpg')
        iid = f'coco_{cid}'
        path = save_img(r['image'], img_dir / f'{iid}.jpg')
        rows_c.append(dict(benchmark='coco_ru', candidate_id=iid, image_path=path))
        for j, txt in enumerate(ru):
            rows_q.append(dict(benchmark='coco_ru', query_id=f'{iid}_ru_{j}', query=txt, positive_id=iid))

    print('coco sampled rows:', len(ds))
    print('coco rows with empty ru:', empty_ru)
    print('coco non-empty ru captions:', total_ru)
    c = pd.DataFrame(rows_c).drop_duplicates('candidate_id')
    q = pd.DataFrame(rows_q)
    q, c = limit(q, c)
    print('coco_ru', len(q), 'queries', len(c), 'candidates')
    return dict(name='coco_ru', q=q, c=c)

def load_vidore():
    if VIDORE_DIR.exists():
        ds = load_from_disk(str(VIDORE_DIR))
        rows = [ds[i] for i in range(min(len(ds), MAX_CANDIDATES if SMOKE_TEST else len(ds)))]
    elif VIDORE_JSONL.exists():
        rows = [json.loads(x) for x in VIDORE_JSONL.read_text(encoding='utf-8').splitlines() if x.strip()]
        if SMOKE_TEST:
            rows = rows[:MAX_CANDIDATES]
    else:
        print('skip vidore')
        return None
    qc, qq = [], []
    for i, r in enumerate(rows):
        path = str(r.get('image_ru_path') or r.get('image') or r.get('original_image_path'))
        txt = str(r.get('query_ru') or r.get('query') or '').strip()
        if not txt or not Path(path).exists():
            continue
        iid = 'vidore_' + str(r.get('row_id', i))
        qc.append(dict(benchmark='vidore_ru_val', candidate_id=iid, image_path=path))
        qq.append(dict(benchmark='vidore_ru_val', query_id=iid + '_q', query=txt, positive_id=iid))
    q, c = limit(pd.DataFrame(qq), pd.DataFrame(qc))
    print('vidore_ru_val', len(q), 'queries', len(c), 'candidates')
    return dict(name='vidore_ru_val', q=q, c=c)

bench = []
for yes, fn in [(RUN_MWS, load_mws), (RUN_COCO, load_coco), (RUN_VIDORE_VAL, load_vidore)]:
    if yes:
        x = fn()
        if x:
            bench.append(x)
print('benchmarks:', [x['name'] for x in bench])


mws 714 queries 373 candidates


Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/28 [00:00<?, ?it/s]

coco:   0%|          | 0/413 [00:00<?, ?it/s]

coco sampled rows: 413
coco rows with empty ru: 0
coco non-empty ru captions: 2066
coco_ru 2066 queries 413 candidates
benchmarks: ['mws', 'coco_ru']


## Metrics

In [6]:
def rr(xs, good):
    return 1 / (xs.index(good) + 1) if good in xs else 0.0

def ndcg(xs, good, k=5):
    return 1 / math.log2(xs[:k].index(good) + 2) if good in xs[:k] else 0.0

def metric(q, ranks):
    rows = []
    for r in q.itertuples(index=False):
        xs = ranks.get(r.query_id, [])
        row = {
            'query_id': r.query_id,
            'positive_id': r.positive_id,
            'MRR': rr(xs, r.positive_id),
            'nDCG@5': ndcg(xs, r.positive_id),
            'top_ids': xs[:TOPK_SAVE],
        }
        for k in [1, 5, 10, 20]:
            hit = int(r.positive_id in xs[:k])
            row[f'Recall@{k}'] = hit
            if k in [1, 5, 10]:
                row[f'Answerable@{k}'] = hit
        rows.append(row)
    per = pd.DataFrame(rows)
    metric_cols = ['MRR', 'nDCG@5', 'Recall@1', 'Recall@5', 'Recall@10', 'Recall@20']
    avg = per[metric_cols].mean().to_dict()
    avg['num_queries'] = len(per)
    return avg, per

## Run model

In [14]:
def to_dev(batch, dev):
    return {k: v.to(dev) if hasattr(v, 'to') else v for k, v in batch.items()}

def remap_colmodern_keys(sd):
    out = {}
    for k, v in sd.items():
        if k.startswith('model.vision_model.vision_model.'):
            k = k.replace('model.vision_model.vision_model.', 'vlm.vision_model.', 1)
        elif k.startswith('model.text_model.'):
            k = k.replace('model.text_model.', 'vlm.text_model.', 1)
        elif k.startswith('model.connector.'):
            k = k.replace('model.connector.', 'vlm.connector.', 1)
        elif k.startswith('custom_text_proj.'):
            k = k.replace('custom_text_proj.', 'embedding_proj_layer.', 1)
        out[k] = v
    return out

def load_model(spec):
    mid = spec.get('base_model') or spec.get('model_id')
    from transformers import AutoTokenizer, Idefics3ImageProcessor, ModernVBertConfig, ColModernVBertConfig, ColModernVBertForRetrieval, ColModernVBertProcessor
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file

    # в transformers сломанный препроцессор, поэтому настраиваем вручную
    image_processor = Idefics3ImageProcessor(
        do_convert_rgb=True,
        do_resize=True,
        size={'longest_edge': 2048},
        do_image_splitting=True,
        max_image_size={'longest_edge': 512},
        do_rescale=True,
        rescale_factor=1 / 255,
        do_normalize=True,
        image_mean=[0.5, 0.5, 0.5],
        image_std=[0.5, 0.5, 0.5],
        do_pad=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(spec.get('tokenizer_dir') or mid)
    proc = ColModernVBertProcessor(image_processor=image_processor, tokenizer=tokenizer, image_seq_len=64)

    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32
    vlm_cfg = ModernVBertConfig.from_pretrained(mid)
    cfg = ColModernVBertConfig(vlm_config=vlm_cfg, embedding_dim=128)
    model = ColModernVBertForRetrieval(cfg)
    if spec.get('tokenizer_dir'):
        model.resize_token_embeddings(len(tokenizer))

    sd = load_file(hf_hub_download(mid, 'model.safetensors'))
    missing, unexpected = model.load_state_dict(remap_colmodern_keys(sd), strict=False)
    missing = [x for x in missing if 'position_ids' not in x]
    if missing or unexpected:
        raise RuntimeError('bad weight load: missing=' + str(missing[:12]) + ' unexpected=' + str(unexpected[:12]))

    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device=dev, dtype=dtype).eval()

    adapter_dir = spec.get('adapter_dir')
    adapter_size = 0
    if adapter_dir:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter_dir).to(device=dev, dtype=dtype).eval()
        adapter_size = sum(p.stat().st_size for p in Path(adapter_dir).rglob('*') if p.is_file())

    stat = dict(model_id=mid, model_memory_bytes=int(model.get_memory_footprint()) if hasattr(model, 'get_memory_footprint') else None, adapter_size_bytes=adapter_size, device=str(dev))
    return proc, model, dev, stat

def split_emb(x, inp): # оставляем отдельный embedding tensor для каждого объекта
    mask = inp.get('attention_mask') if hasattr(inp, 'get') else None
    if mask is None:
        return [x[i].detach().cpu() for i in range(x.shape[0])]
    return [x[i, mask[i].bool()].detach().cpu() for i in range(x.shape[0])]

def img_emb(b, proc, model, dev, mkey, stat): # кодируем и кэшируем картинки-кандидаты
    cache_dir = emb_dir / b['name'] / key(mkey)
    cache_dir.mkdir(parents=True, exist_ok=True)

    emb_path = cache_dir / 'image_embeddings.pt'
    ids_path = cache_dir / 'candidate_ids.json'
    candidate_ids = b['c']['candidate_id'].tolist()

    if emb_path.exists() and ids_path.exists() and json.loads(ids_path.read_text()) == candidate_ids:
        stat[b['name'] + '_index_size_bytes'] = emb_path.stat().st_size
        return torch.load(emb_path, map_location='cpu')

    image_embeddings = []
    t0 = time.perf_counter()
    image_paths = b['c']['image_path'].tolist()

    for start in tqdm(range(0, len(image_paths), IMAGE_BATCH_SIZE), desc='images ' + b['name']):
        batch_paths = image_paths[start:start + IMAGE_BATCH_SIZE]
        images = [Image.open(p).convert('RGB') for p in batch_paths]
        texts = ['<image>'] * len(images)
        batch = proc(text=texts, images=images, return_tensors='pt')
        batch = to_dev(batch, dev)

        with torch.inference_mode():
            out = model(**batch).embeddings
        image_embeddings.extend(split_emb(out, batch))

    torch.save(image_embeddings, emb_path)
    ids_path.write_text(json.dumps(candidate_ids, ensure_ascii=False, indent=2), encoding='utf-8')

    stat[b['name'] + '_document_latency_sec'] = (time.perf_counter() - t0) / max(1, len(image_paths))
    stat[b['name'] + '_index_size_bytes'] = emb_path.stat().st_size
    return image_embeddings

# кодируем запросы и ранжируем картинки
def rank(b, image_embeddings, proc, model, dev, stat):
    candidate_ids = b['c']['candidate_id'].tolist()
    image_embeddings = [x.to(dev) for x in image_embeddings]

    rankings = {}
    encode_time = 0
    search_time = 0
    query_count = 0

    for start in tqdm(range(0, len(b['q']), QUERY_BATCH_SIZE), desc='queries ' + b['name']):
        batch_q = b['q'].iloc[start:start + QUERY_BATCH_SIZE]
        texts = batch_q['query'].astype(str).tolist()

        t0 = time.perf_counter()
        batch = proc(text=texts, return_tensors='pt')
        batch = to_dev(batch, dev)
        with torch.inference_mode():
            query_embeddings = split_emb(model(**batch).embeddings, batch)
            query_embeddings = [x.to(dev) for x in query_embeddings]
        encode_time += time.perf_counter() - t0

        t0 = time.perf_counter()
        with torch.inference_mode():
            scores = proc.score_retrieval(query_embeddings=query_embeddings, passage_embeddings=image_embeddings)
        search_time += time.perf_counter() - t0

        order = torch.argsort(scores.detach().cpu(), dim=1, descending=True).numpy()
        for query_id, candidate_order in zip(batch_q['query_id'].tolist(), order):
            rankings[query_id] = [candidate_ids[i] for i in candidate_order[:TOPK_SAVE]]

        query_count += len(batch_q)

    stat[b['name'] + '_query_latency_sec'] = encode_time / max(1, query_count)
    stat[b['name'] + '_retrieval_latency_sec'] = search_time / max(1, query_count)
    return rankings


def mean_pool(last_hidden, mask):
    mask = mask.unsqueeze(-1).to(last_hidden.dtype)
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1)

def encode_dense_texts(texts, tokenizer, model, dev, prefix):
    all_vecs = []
    for start in tqdm(range(0, len(texts), TEXT_BATCH_SIZE), desc='dense text'):
        batch_texts = [prefix + str(x) for x in texts[start:start + TEXT_BATCH_SIZE]]
        batch = tokenizer(batch_texts, padding=True, truncation=True, return_tensors='pt', max_length=512)
        batch = to_dev(batch, dev)
        with torch.inference_mode():
            out = model(**batch).last_hidden_state
            vec = mean_pool(out, batch['attention_mask'])
            vec = torch.nn.functional.normalize(vec, dim=1)
        all_vecs.append(vec.detach().cpu())
    return torch.cat(all_vecs) if all_vecs else torch.empty(0)

def rank_ocr_dense(b, spec, stat):
    cache = spec.get('ocr_cache') or OCR_DENSE_CACHE
    if not cache:
        raise ValueError('OCR-DENSE needs ocr_cache path')
    docs = pd.read_parquet(cache)
    docs = docs[['candidate_id', 'ocr_text']].drop_duplicates('candidate_id')
    docs = b['c'][['candidate_id']].merge(docs, on='candidate_id', how='left').fillna({'ocr_text': ''})

    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    text_model = spec.get('text_model') or OCR_DENSE_MODEL
    tok = AutoTokenizer.from_pretrained(text_model)
    model = AutoModel.from_pretrained(text_model).to(dev).eval()

    t0 = time.perf_counter()
    doc_vec = encode_dense_texts(docs['ocr_text'].tolist(), tok, model, dev, 'passage: ')
    stat[b['name'] + '_document_latency_sec'] = (time.perf_counter() - t0) / max(1, len(docs))
    stat[b['name'] + '_index_size_bytes'] = int(doc_vec.numpy().nbytes)

    t0 = time.perf_counter()
    query_vec = encode_dense_texts(b['q']['query'].astype(str).tolist(), tok, model, dev, 'query: ')
    stat[b['name'] + '_query_latency_sec'] = (time.perf_counter() - t0) / max(1, len(b['q']))

    t0 = time.perf_counter()
    scores = query_vec @ doc_vec.T
    stat[b['name'] + '_retrieval_latency_sec'] = (time.perf_counter() - t0) / max(1, len(b['q']))
    stat.update(model_id=text_model, model_memory_bytes=int(model.get_memory_footprint()) if hasattr(model, 'get_memory_footprint') else None, adapter_size_bytes=0, device=str(dev))

    ids = docs['candidate_id'].tolist()
    rankings = {}
    order = torch.argsort(scores, dim=1, descending=True).numpy()
    for qid, idx in zip(b['q']['query_id'].tolist(), order):
        rankings[qid] = [ids[j] for j in idx[:TOPK_SAVE]]
    return rankings

def make_eff_rows(stat, bname, mid):
    def row(metric, meaning, value, unit):
        return {
            'Metric': metric,
            'Meaning': meaning,
            'Value': value,
            'Unit': unit,
            'Benchmark': bname,
            'Model ID': mid,
        }

    return [
        row('Model size', 'full checkpoint size', stat.get('model_memory_bytes'), 'bytes'),
        row('Adapter size', 'LoRA/additional-token parameter size', stat.get('adapter_size_bytes', 0), 'bytes'),
        row('Index size', 'stored document embeddings', stat.get(bname + '_index_size_bytes'), 'bytes'),
        row('Query latency', 'time to encode one query', stat.get(bname + '_query_latency_sec'), 'seconds/query'),
        row('Document/image latency', 'time to encode one page/image', stat.get(bname + '_document_latency_sec'), 'seconds/item'),
        row('Retrieval latency', 'search time over candidate pool', stat.get(bname + '_retrieval_latency_sec'), 'seconds/query'),
        row('CPU/GPU setting', 'device used for inference', stat.get('device'), 'device'),
    ]


## Go

In [15]:
import hashlib

if not bench:
    raise RuntimeError('no data loaded')

result_cache = root / 'cache' / 'retrieval_results'
result_cache.mkdir(parents=True, exist_ok=True)

def model_run_key(spec, model_label):
    payload = {k: spec.get(k) for k in ['id', 'kind', 'model_id', 'base_model', 'adapter_dir', 'tokenizer_dir', 'text_model', 'ocr_cache']}
    h = hashlib.sha1(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:12]
    return key(model_label + '_' + h)

def run_cache_dir(spec, b, model_label):
    payload = {
        'spec': {k: spec.get(k) for k in ['id', 'kind', 'model_id', 'base_model', 'adapter_dir', 'tokenizer_dir', 'text_model', 'ocr_cache']},
        'benchmark': b['name'],
        'topk': TOPK_SAVE,
        'queries': b['q'][['query_id', 'query', 'positive_id']].astype(str).values.tolist(),
        'candidates': b['c']['candidate_id'].astype(str).tolist(),
    }
    h = hashlib.sha1(json.dumps(payload, ensure_ascii=False, sort_keys=True).encode()).hexdigest()[:16]
    return result_cache / key(spec['id']) / b['name'] / h

def clean_per_query(df):
    df = df.copy()
    for col in ['experiment_id', 'model_id', 'benchmark', 'query_id', 'positive_id']:
        if col in df:
            df[col] = df[col].astype(str)
    if 'top_ids' in df:
        df['top_ids'] = df['top_ids'].map(lambda x: x if isinstance(x, str) else json.dumps(x, ensure_ascii=False))
    return df

def load_cached(cache_dir):
    need = [cache_dir / 'metric.json', cache_dir / 'per_query.parquet', cache_dir / 'efficiency.json']
    if not all(p.exists() for p in need):
        return None
    metric_row = json.loads((cache_dir / 'metric.json').read_text(encoding='utf-8'))
    per_query = pd.read_parquet(cache_dir / 'per_query.parquet')
    eff_rows_cached = json.loads((cache_dir / 'efficiency.json').read_text(encoding='utf-8'))
    return metric_row, per_query, eff_rows_cached

def save_cached(cache_dir, metric_row, per_query, eff_rows_data, rankings):
    cache_dir.mkdir(parents=True, exist_ok=True)
    (cache_dir / 'metric.json').write_text(json.dumps(metric_row, ensure_ascii=False, indent=2), encoding='utf-8')
    clean_per_query(per_query).to_parquet(cache_dir / 'per_query.parquet', index=False)
    (cache_dir / 'efficiency.json').write_text(json.dumps(eff_rows_data, ensure_ascii=False, indent=2), encoding='utf-8')
    (cache_dir / 'rankings.json').write_text(json.dumps(rankings, ensure_ascii=False), encoding='utf-8')

def write_outputs():
    metrics = pd.DataFrame(metric_rows)
    per_query = pd.concat(per_query_tables, ignore_index=True) if per_query_tables else pd.DataFrame()
    per_query = clean_per_query(per_query) if len(per_query) else per_query
    eff = pd.DataFrame(efficiency_rows)
    metrics.to_csv(out / 'metrics_summary.csv', index=False)
    (out / 'metrics_summary.json').write_text(metrics.to_json(orient='records', force_ascii=False, indent=2), encoding='utf-8')
    per_query.to_parquet(out / 'per_query_results.parquet', index=False)
    eff.to_csv(out / 'efficiency_summary.csv', index=False)
    return metrics, per_query, eff

metric_rows = []
per_query_tables = []
efficiency_rows = []

for spec in models:
    if not spec.get('enabled', True):
        continue
    model_label = spec.get('model_id') or spec.get('base_model') or spec.get('text_model')
    print('run', spec['id'], model_label)

    jobs = []
    for b in bench:
        if spec.get('kind') == 'ocr_dense' and b['name'] != 'mws':
            continue
        cache_dir = run_cache_dir(spec, b, model_label)
        cached = load_cached(cache_dir)
        if cached:
            metric_row, per_query, eff_rows_cached = cached
            metric_rows.append(metric_row)
            per_query_tables.append(per_query)
            efficiency_rows += eff_rows_cached
            print('cache hit', spec['id'], b['name'])
            display(pd.DataFrame([metric_row]))
            write_outputs()
        else:
            jobs.append((b, cache_dir))

    if not jobs:
        continue

    if spec.get('kind') == 'ocr_dense':
        proc = model = dev = None
        stat = {}
    else:
        proc, model, dev, stat = load_model(spec)
        emb_key = model_run_key(spec, model_label)

    for b, cache_dir in jobs:
        if spec.get('kind') == 'ocr_dense':
            rankings = rank_ocr_dense(b, spec, stat)
        else:
            image_embeddings = img_emb(b, proc, model, dev, emb_key, stat)
            rankings = rank(b, image_embeddings, proc, model, dev, stat)

        metric_row, per_query = metric(b['q'], rankings)
        metric_row = dict(experiment_id=spec['id'], model_id=model_label, benchmark=b['name'], **metric_row)
        per_query.insert(0, 'benchmark', b['name'])
        per_query.insert(0, 'model_id', model_label)
        per_query.insert(0, 'experiment_id', spec['id'])

        eff_rows_now = make_eff_rows(stat, b['name'], model_label) if RUN_EFFICIENCY else []
        save_cached(cache_dir, metric_row, per_query, eff_rows_now, rankings)

        metric_rows.append(metric_row)
        per_query_tables.append(clean_per_query(per_query))
        efficiency_rows += eff_rows_now

        display(pd.DataFrame([metric_row]))
        write_outputs()

    if model is not None:
        del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

metrics, per_query, eff = write_outputs()

display(metrics)

if len(eff):
    eff_pretty = eff.pivot_table(
        index=['Benchmark', 'Model ID'],
        columns='Metric',
        values='Value',
        aggfunc='first'
    ).reset_index()
    display(eff_pretty)
else:
    display(eff)

print('saved to', out)
print('cache:', result_cache)


run ZS-MM ModernVBERT/colmodernvbert-merged
cache hit ZS-MM mws


,experiment_id,model_id,benchmark,MRR,nDCG@5,Recall@1,Recall@5,Recall@10,Recall@20,num_queries
0,ZS-MM,ModernVBERT/colmodernvbert-merged,mws,0.023943,0.021,0.011204,0.030812,0.057423,0.095238,714


cache hit ZS-MM coco_ru


,experiment_id,model_id,benchmark,MRR,nDCG@5,Recall@1,Recall@5,Recall@10,Recall@20,num_queries
0,ZS-MM,ModernVBERT/colmodernvbert-merged,coco_ru,0.023766,0.021035,0.00726,0.03485,0.06486,0.113746,2066


run RU-LoRA-L2-H2 ModernVBERT/colmodernvbert-merged


...

dense text:   0%|          | 0/45 [00:00<?, ?it/s]

,experiment_id,model_id,benchmark,MRR,nDCG@5,Recall@1,Recall@5,Recall@10,Recall@20,num_queries
0,OCR-DENSE,intfloat/multilingual-e5-large,mws,0.177604,0.176897,0.135854,0.215686,0.281513,0.348739,714


,experiment_id,model_id,benchmark,MRR,nDCG@5,Recall@1,Recall@5,Recall@10,Recall@20,num_queries
0,ZS-MM,ModernVBERT/colmodernvbert-merged,mws,0.023943,0.021000,0.011204,0.030812,0.057423,0.095238,714
1,ZS-MM,ModernVBERT/colmodernvbert-merged,coco_ru,0.023766,0.021035,0.007260,0.034850,0.064860,0.113746,2066
2,RU-LoRA-L2-H2,ModernVBERT/colmodernvbert-merged,mws,0.043301,0.041057,0.023810,0.058824,0.092437,0.144258,714
3,RU-LoRA-L2-H2,ModernVBERT/colmodernvbert-merged,coco_ru,0.063966,0.061594,0.026137,0.097773,0.155373,0.250726,2066
4,RU-LoRA-R8-L2,ModernVBERT/colmodernvbert-merged,mws,0.037155,0.032716,0.019608,0.047619,0.082633,0.142857,714
5,RU-LoRA-R8-L2,ModernVBERT/colmodernvbert-merged,coco_ru,0.061436,0.058682,0.025653,0.091965,0.149080,0.238625,2066
6,OCR-DENSE,intfloat/multilingual-e5-large,mws,0.177604,0.176897,0.135854,0.215686,0.281513,0.348739,714


Metric,Benchmark,Model ID,Adapter size,CPU/GPU setting,Document/image latency,Index size,Model size,Query latency,Retrieval latency
0,coco_ru,ModernVBERT/colmodernvbert-merged,0,cuda,1.86107,94670528,504209920,0.007861,0.002603
1,mws,ModernVBERT/colmodernvbert-merged,0,cuda,1.968664,87195688,504209920,0.032561,0.012634
2,mws,intfloat/multilingual-e5-large,0,cuda,0.095746,1527808,2239569952,0.029668,0.000061


saved to /content/drive/MyDrive/experiments_vbert/runs/20260512_113733
cache: /content/drive/MyDrive/experiments_vbert/cache/retrieval_results
